# 02 · Kronos bias diagnosis (Colab T4)

Runtime → Change runtime type → **T4 GPU**. After any disconnect, rerun **Setup** only, then the step you were on.
Sampled paths and reports live on Drive (`MyDrive/kronos-agent/`), so nothing is lost on reset.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/kronos-agent'
CODE = f'{ROOT}/code'
KRONOS = '/content/Kronos'

# Unpack code from Drive. Set UPDATE = True once after uploading a new kronos-agent.zip.
import os, shutil, zipfile, sys
UPDATE = False
ZIP = f'{ROOT}/kronos-agent.zip'
if UPDATE or not os.path.exists(f'{CODE}/scripts/diagnose_kronos.py'):
    assert os.path.exists(ZIP), f'upload kronos-agent.zip to {ROOT}/ first'
    shutil.rmtree(CODE, ignore_errors=True)
    zipfile.ZipFile(ZIP).extractall(CODE)

!test -d $KRONOS || git clone -q https://github.com/shiyu-coder/Kronos $KRONOS
%pip install -q einops safetensors ccxt yfinance
%pip install -q -e "$CODE" --no-deps
if CODE not in sys.path:
    sys.path.insert(0, CODE)
!nvidia-smi -L
print(ROOT, CODE, KRONOS)

## Step 1 · Tokenizer round-trip test (GPU, ~1 min)
Does encode→decode lose price level when the last price is far from the context mean?

In [ ]:
!cd $CODE && python scripts/diagnose_kronos.py --config configs/spy_1d.yaml --root $ROOT --kronos-path $KRONOS --n 60

## Step 2 · SPY Phase 2 with EMOS correction (CPU, ~2 min)

In [ ]:
!cd $CODE && python scripts/run_phase2.py --config configs/spy_1d.yaml --root $ROOT

## Step 3 · Lookback sweep on SPY 2025 (GPU, ~15–20 min)

In [ ]:
import sys, subprocess, pandas as pd
for k in [k for k in sys.modules if k.startswith('kronos_agent')]: del sys.modules[k]
from kronos_agent.config import load_config, run_tag, dm_horizon
from kronos_agent.data import load_data
from kronos_agent.store import ResultStore
from kronos_agent.evaluate import build_eval_frame
from kronos_agent.phase2 import phase2_report

W = ['windows.pre_cutoff=null', 'windows.post_cutoff=[2025-01-01,2026-01-01]']
out = []
for L in (64, 128, 256):
    sets = [f'lookback={L}'] + W
    r = subprocess.run(['python', 'scripts/run_sampling.py', '--config', 'configs/spy_1d.yaml', '--root', ROOT,
                        '--kronos-path', KRONOS] + sum((['--set', s] for s in sets), []),
                       cwd=CODE, capture_output=True, text=True)
    print(f'L={L}:', r.stdout.strip().splitlines()[-1] if r.returncode == 0 else r.stderr[-800:])
    cfg = load_config(f'{CODE}/configs/spy_1d.yaml', sets)
    s = ResultStore(f'{ROOT}/paths/{run_tag(cfg)}').load()
    s = s[(s.origin_ts >= '2025-01-01') & (s.origin_ts < '2026-01-01')].assign(period='post_cutoff')
    ev = build_eval_frame(s, load_data(cfg, f'{ROOT}/data'), L, seed=cfg['seed'])
    rep = phase2_report(ev, cfg, dm_horizon(cfg))
    b = rep['bias'].query("base == 'kronos'")
    c = rep['calibration'].query("base == 'kronos' and level != level")[['method', 'crps']]
    out.append(b.merge(c, left_on='dist', right_on='method').assign(L=L))
print(pd.concat(out)[['L', 'dist', 'n', 'bias', 'mae', 'crps', 'ic', 'dir_acc', 'slope_mu_gap']].round(4).to_string(index=False))